<a href="https://colab.research.google.com/github/mcovarrubiaz-hue/Chatbot_LCC/blob/main/Descomposici%C3%B3n_de_ra%C3%ADces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# @title
import random
import math
import ipywidgets as widgets
from IPython.display import display

# Estilos visuales CSS
card_style = "width: 140px; height: 100px; background-color: white; border: 2px solid #2c3e50; border-radius: 10px; box-shadow: 3px 3px 8px rgba(0,0,0,0.5); display: flex; justify-content: center; align-items: center;"

# Estado del juego
estado = {"fase": 1, "raices": [], "coeficientes": [], "signos": [], "base": 1, "html_pasos": "", "html_resultado": ""}

# Botones (Reordenados y con márgenes ajustados)
boton_accion = widgets.Button(description='🃏 Repartir Nuevas Raíces', button_style='success', layout=widgets.Layout(width='220px', height='50px', margin='0 5px 0 0'))
boton_resultado = widgets.Button(description='✅ Ver Resultado', button_style='info', disabled=True, layout=widgets.Layout(width='220px', height='50px', margin='0 5px 0 0'))
boton_pasos = widgets.Button(description='🔍 Ver Pasos', button_style='warning', disabled=True, layout=widgets.Layout(width='220px', height='50px'))

# El botón amarillo (pasos) ahora está al final de la lista
caja_botones = widgets.HBox([boton_accion, boton_resultado, boton_pasos], layout=widgets.Layout(justify_content='center', margin='0 0 20px 0'))
etiqueta_resultado = widgets.HTML(value="<h3 style='text-align:center; color:#7f8c8d; font-family: Helvetica;'>Presiona 'Repartir Nuevas Raíces' para iniciar...</h3>")

# Función para formatear el símbolo de raíz cuadrada en HTML
def formato_raiz(numero, color_texto="#2c3e50"):
    return f'<span style="white-space: nowrap; font-size: 38px; font-weight: bold; color: {color_texto};">&radic;<span style="text-decoration:overline;">&nbsp;{numero}&nbsp;</span></span>'

# Función para formatear el resultado con coeficientes
def formato_resultado_raiz(coef, base):
    if coef == 1:
        return formato_raiz(base, "#e74c3c")
    elif coef == -1:
        return f'<span style="color: #e74c3c; font-weight: bold; font-size: 45px;">-</span>{formato_raiz(base, "#e74c3c")}'
    elif coef == 0:
        return '<span style="color: #e74c3c; font-weight: bold; font-size: 45px;">0</span>'
    else:
        return f'<span style="color: #e74c3c; font-weight: bold; font-size: 45px;">{coef}</span>{formato_raiz(base, "#e74c3c")}'

# Función que dibuja la mesa completa dinámicamente
def generar_mesa_html(raices, signos, fase, html_pasos="", html_resultado=""):
    cartas_html = ""
    for i in range(len(raices)):
        cartas_html += f'<div style="{card_style}">{formato_raiz(raices[i])}</div>'
        if i < len(raices) - 1:
            sig = "?" if fase == 1 else signos[i]
            color_sig = "#7f8c8d" if fase == 1 else ("#2ecc71" if sig == '+' else "#e74c3c")
            cartas_html += f'''
            <div style="font-size: 55px; font-weight: bold; color: {color_sig}; text-shadow: 2px 2px 5px rgba(0,0,0,0.4); margin: 0 10px;">
                {sig}
            </div>'''

    texto_op = "¡DESCOMPONER RAÍCES!" if fase == 1 else "¡REDUCIR TÉRMINOS SEMEJANTES!"
    color_op = "#7f8c8d" if fase == 1 else "#e67e22"

    html = f"""
    <div style="background-color: #2980b9; padding: 30px; border-radius: 15px; display: flex; flex-direction: column; align-items: center; box-shadow: inset 0 0 20px rgba(0,0,0,0.4); font-family: Helvetica, Arial, sans-serif; min-width: 650px;">

        <div style="font-size: 26px; font-weight: bold; color: {color_op}; background-color: rgba(255,255,255,0.95); padding: 10px 25px; border-radius: 10px; margin-bottom: 25px; box-shadow: 2px 2px 5px rgba(0,0,0,0.3);">
            {texto_op}
        </div>

        <div style="display: flex; justify-content: center; align-items: center; flex-wrap: wrap; width: 100%;">
            {cartas_html}
        </div>
    """

    if html_resultado:
        html += html_resultado

    if html_pasos:
        html += html_pasos

    html += "</div>"
    return html

# Lógica del clic principal
def manejar_clic_accion(b):
    if estado["fase"] == 1:
        num_raices = random.choice([2, 3, 4])
        base = random.choice([2, 3, 5, 6, 7, 10, 11])
        estado["base"] = base

        max_sq = 1000 // base
        max_root = int(math.sqrt(max_sq))

        opciones_coeficientes = list(range(2, max_root + 1))
        coeficientes = random.sample(opciones_coeficientes, num_raices)
        estado["coeficientes"] = coeficientes
        estado["raices"] = [(c**2) * base for c in coeficientes]
        estado["signos"] = [random.choice(['+', '-']) for _ in range(num_raices - 1)]

        estado["html_pasos"] = ""
        estado["html_resultado"] = ""

        etiqueta_resultado.value = generar_mesa_html(estado["raices"], estado["signos"], 1)

        boton_accion.description = "⚠️ Revelar Operación"
        boton_accion.button_style = "danger"
        boton_pasos.disabled = True
        boton_resultado.disabled = True
        estado["fase"] = 2

    else:
        etiqueta_resultado.value = generar_mesa_html(estado["raices"], estado["signos"], 2)

        boton_accion.description = "🃏 Repartir Nuevas Raíces"
        boton_accion.button_style = "success"
        boton_pasos.disabled = False
        boton_resultado.disabled = False
        estado["fase"] = 1

# Lógica para mostrar el resultado final
def manejar_clic_resultado(b):
    coef_final = estado["coeficientes"][0]
    for i, sig in enumerate(estado["signos"]):
        if sig == "+":
            coef_final += estado["coeficientes"][i+1]
        elif sig == "-":
            coef_final -= estado["coeficientes"][i+1]

    texto_res = formato_resultado_raiz(coef_final, estado["base"])

    estado["html_resultado"] = f"""
    <div style="margin-top: 20px; background-color: white; padding: 15px 40px; border-radius: 10px; color: #2c3e50; box-shadow: 3px 3px 10px rgba(0,0,0,0.5); width: 85%; display: flex; align-items: center;">
        <h4 style="margin: 0 20px 0 0; font-size: 26px;">Resultado Final:</h4>
        <div>{texto_res}</div>
    </div>
    """

    etiqueta_resultado.value = generar_mesa_html(estado["raices"], estado["signos"], 2, estado["html_pasos"], estado["html_resultado"])
    boton_resultado.disabled = True

# Lógica para mostrar los pasos
def manejar_clic_pasos(b):
    pasos_str = ""
    for r, c in zip(estado["raices"], estado["coeficientes"]):
        base = estado["base"]
        sq = c**2
        linea = f"{formato_raiz(r)} &nbsp;=&nbsp; {formato_raiz(f'{sq} &middot; {base}')} &nbsp;=&nbsp; <span style='font-size:38px; font-weight:bold;'>{c}</span>{formato_raiz(base)}"
        pasos_str += f"<div style='margin-bottom: 12px; display: flex; align-items: center;'>{linea}</div>"

    estado["html_pasos"] = f"""
    <div style="margin-top: 20px; background-color: #f39c12; padding: 20px 40px; border-radius: 10px; color: #2c3e50; box-shadow: 3px 3px 10px rgba(0,0,0,0.5); width: 85%;">
        <h4 style="margin: 0 0 15px 0; font-size: 24px;">Descomposición paso a paso:</h4>
        <div style="display: flex; flex-direction: column; align-items: flex-start; padding-left: 20px;">
            {pasos_str}
        </div>
    </div>
    """

    etiqueta_resultado.value = generar_mesa_html(estado["raices"], estado["signos"], 2, estado["html_pasos"], estado["html_resultado"])
    boton_pasos.disabled = True

# Conectar eventos
boton_accion.on_click(manejar_clic_accion)
boton_resultado.on_click(manejar_clic_resultado)
boton_pasos.on_click(manejar_clic_pasos)

# Mostrar todo en pantalla
display(caja_botones, etiqueta_resultado)

HTML(value="<h3 style='text-align:center; color:#7f8c8d; font-family: Helvetica;'>Presiona 'Repartir Nuevas Ra…